In [19]:
# Load and inspect
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/flipkart_com-ecommerce_sample.csv')
print(df.shape)
print(df.columns.tolist())
df.head(3)

(20000, 15)
['uniq_id', 'crawl_timestamp', 'product_url', 'product_name', 'product_category_tree', 'pid', 'retail_price', 'discounted_price', 'image', 'is_FK_Advantage_product', 'description', 'product_rating', 'overall_rating', 'brand', 'product_specifications']


,uniq_id,crawl_timestamp,product_url,product_name,product_category_tree,pid,retail_price,discounted_price,image,is_FK_Advantage_product,description,product_rating,overall_rating,brand,product_specifications
0,c2d766ca982eca8304150849735ffef9,2016-03-25 22:59:23 +0000,http://www.flipkart.com/alisha-solid-women-s-c...,Alisha Solid Women's Cycling Shorts,"[""Clothing >> Women's Clothing >> Lingerie, Sl...",SRTEH2FF9KEDEFGF,999.0,379.0,"[""http://img5a.flixcart.com/image/short/u/4/a/...",False,Key Features of Alisha Solid Women's Cycling S...,No rating available,No rating available,Alisha,"{""product_specification""=>[{""key""=>""Number of ..."
1,7f7036a6d550aaa89d34c77bd39a5e48,2016-03-25 22:59:23 +0000,http://www.flipkart.com/fabhomedecor-fabric-do...,FabHomeDecor Fabric Double Sofa Bed,"[""Furniture >> Living Room Furniture >> Sofa B...",SBEEH3QGU7MFYJFY,32157.0,22646.0,"[""http://img6a.flixcart.com/image/sofa-bed/j/f...",False,FabHomeDecor Fabric Double Sofa Bed (Finish Co...,No rating available,No rating available,FabHomeDecor,"{""product_specification""=>[{""key""=>""Installati..."
2,f449ec65dcbc041b6ae5e6a32717d01b,2016-03-25 22:59:23 +0000,http://www.flipkart.com/aw-bellies/p/itmeh4grg...,AW Bellies,"[""Footwear >> Women's Footwear >> Ballerinas >...",SHOEH4GRSUBJGZXE,999.0,499.0,"[""http://img5a.flixcart.com/image/shoe/7/z/z/r...",False,Key Features of AW Bellies Sandals Wedges Heel...,No rating available,No rating available,AW,"{""product_specification""=>[{""key""=>""Ideal For""..."


In [20]:
# Clean prices
def parse_price(val):
    """Parse price strings like '₹1,299' or '1299.0' to float."""
    if pd.isna(val):
        return np.nan
    s = str(val).replace('₹', '').replace(',', '').strip()
    try:
        return float(s)
    except:
        return np.nan

df['retail_price']     = df['retail_price'].apply(parse_price)
df['discounted_price'] = df['discounted_price'].apply(parse_price)

# Drop rows with missing or invalid prices
df = df.dropna(subset=['retail_price', 'discounted_price'])
df = df[df['retail_price'] > 0]
df = df[df['discounted_price'] > 0]

# Sanity: discounted_price must be <= retail_price (MRP rule)
df = df[df['discounted_price'] <= df['retail_price']]

# Remove extreme outliers (top/bottom 1%)
lo = df['discounted_price'].quantile(0.01)
hi = df['discounted_price'].quantile(0.99)
df = df[(df['discounted_price'] >= lo) & (df['discounted_price'] <= hi)]

print(f"Clean rows: {len(df):,}")
print(f"Price range: ₹{df['discounted_price'].min():,.0f} – ₹{df['discounted_price'].max():,.0f}")
print(f"Median price: ₹{df['discounted_price'].median():,.0f}")

Clean rows: 19,525
Price range: ₹120 – ₹31,916
Median price: ₹550


In [21]:
#Extract category
def extract_category(tree):
    """
    Input:  '["Clothing >> Kids' Clothing >> Girls' Clothing >> Skirts"]'
    Output: 'Clothing'
    """
    if pd.isna(tree):
        return 'unknown'
    try:
        # Remove brackets/quotes, split on >>
        clean = str(tree).strip('[]"\'')
        parts = [p.strip() for p in clean.split('>>')]
        return parts[0].lower().replace(" ", "_") if parts else 'unknown'
    except:
        return 'unknown'

df['category'] = df['product_category_tree'].apply(extract_category)

# Keep top 15 categories (enough variety, enough samples)
top_cats = df['category'].value_counts().head(15).index.tolist()
df = df[df['category'].isin(top_cats)]
print(f"Categories kept: {len(top_cats)}")
print(df['category'].value_counts().head(10))

Categories kept: 15
category
clothing                      6118
jewellery                     3355
footwear                      1225
mobiles_&_accessories         1085
automotive                    1008
home_decor_&_festive_needs     923
beauty_and_personal_care       702
home_furnishing                696
kitchen_&_dining               637
computers                      555
Name: count, dtype: int64


In [22]:
#Build competitor prices
# For each product, competitors = other products in same category
# Sort by discounted_price within category, find nearest 3 neighbors by price

def add_competitor_prices(df):
    df = df.copy().reset_index(drop=True)
    
    comp_1_list, comp_2_list, comp_3_list = [], [], []
    
    for cat in df['category'].unique():
        cat_mask = df['category'] == cat
        cat_df   = df[cat_mask].copy()
        prices   = cat_df['discounted_price'].values
        
        for i, (idx, row) in enumerate(cat_df.iterrows()):
            focal_price = row['discounted_price']
            # All prices except focal product
            other = np.delete(prices, i)
            
            if len(other) < 3:
                # Not enough competitors — use retail_price * noise as fallback
                comps = [focal_price * np.random.uniform(0.85, 1.15) for _ in range(3)]
            else:
                # Nearest 3 competitors by absolute price difference
                diffs  = np.abs(other - focal_price)
                nearest = other[np.argsort(diffs)[:3]]
                comps  = sorted(nearest.tolist())
            
            comp_1_list.append(comps[0])
            comp_2_list.append(comps[1])
            comp_3_list.append(comps[2])
    
    # Map back by original index (loop builds in category order, need to realign)
    # Simpler: process all at once in original df order
    return comp_1_list, comp_2_list, comp_3_list

# Rebuild cleanly — process in original df order
df_sorted = df.sort_values('category').reset_index(drop=True)
comp_1, comp_2, comp_3 = [], [], []

for cat in df_sorted['category'].unique():
    cat_df = df_sorted[df_sorted['category'] == cat].copy()
    prices = cat_df['discounted_price'].values
    
    for i in range(len(cat_df)):
        fp    = prices[i]
        other = np.delete(prices, i)
        if len(other) < 3:
            comps = [fp * np.random.uniform(0.88, 1.12) for _ in range(3)]
        else:
            diffs   = np.abs(other - fp)
            nearest = other[np.argsort(diffs)[:3]]
            comps   = sorted(nearest.tolist())
        comp_1.append(comps[0])
        comp_2.append(comps[1])
        comp_3.append(comps[2])

df_sorted['comp_1'] = comp_1
df_sorted['comp_2'] = comp_2
df_sorted['comp_3'] = comp_3

df = df_sorted.copy()
print(f"Competitor prices added. Sample:")
print(df[['discounted_price','comp_1','comp_2','comp_3']].head(5))

Competitor prices added. Sample:
   discounted_price  comp_1  comp_2  comp_3
0            1599.0  1599.0  1599.0  1599.0
1             269.0   278.0   278.0   278.0
2             920.0   920.0   920.0   920.0
3            1850.0  1850.0  1850.0  1850.0
4            1599.0  1599.0  1599.0  1599.0


In [23]:
# Clean rating + brand
def parse_rating(val):
    try:
        r = float(str(val).strip())
        return r if 1.0 <= r <= 5.0 else 4.0
    except:
        return 4.0

df['rating'] = df['product_rating'].apply(parse_rating)

# Encode brand: top 50 brands get ID, rest = 0
top_brands = df['brand'].value_counts().head(50).index.tolist()
brand_map  = {b: i+1 for i, b in enumerate(top_brands)}
df['brand_encoded'] = df['brand'].map(brand_map).fillna(0).astype(int)

print(f"Brands encoded: {len(brand_map)} top brands")
print(df['rating'].describe())

Brands encoded: 50 top brands
count    18269.000000
mean         3.983803
std          0.385570
min          1.000000
25%          4.000000
50%          4.000000
75%          4.000000
max          5.000000
Name: rating, dtype: float64


In [24]:
#Engineer features
def engineer_features(df):
    d = df.copy()
    
    comp_avg = (d['comp_1'] + d['comp_2'] + d['comp_3']) / 3
    comp_min = d[['comp_1','comp_2','comp_3']].min(axis=1)
    comp_max = d[['comp_1','comp_2','comp_3']].max(axis=1)
    
    d['comp_avg_price']   = comp_avg
    d['comp_min_price']   = comp_min
    d['comp_max_price']   = comp_max
    d['price_vs_comp']    = d['discounted_price'] / (comp_avg + 1e-9)
    d['price_gap']        = d['discounted_price'] - comp_min      # gap to cheapest competitor
    d['discount_pct']     = (d['retail_price'] - d['discounted_price']) / d['retail_price']
    d['log_retail']       = np.log1p(d['retail_price'])
    d['log_discounted']   = np.log1p(d['discounted_price'])
    d['price_per_rating'] = d['discounted_price'] / (d['rating'] + 0.1)
    d['margin_proxy']     = d['discounted_price'] / (d['retail_price'] + 1e-9)
    d['comp_spread']      = comp_max - comp_min                   # competitor price spread
    d['above_comp_avg']   = (d['discounted_price'] > comp_avg).astype(int)
    
    return d

df = engineer_features(df)

FEATURES = [
    'comp_avg_price',
    'comp_min_price',
    'comp_max_price',
    'price_vs_comp',
    'price_gap',
    'discount_pct',
    'log_retail',
    'price_per_rating',
    'margin_proxy',
    'comp_spread',
    'above_comp_avg',
    'rating',
    'brand_encoded',
    'category_encoded',   # will add next
]

print("Features ready:", FEATURES)

Features ready: ['comp_avg_price', 'comp_min_price', 'comp_max_price', 'price_vs_comp', 'price_gap', 'discount_pct', 'log_retail', 'price_per_rating', 'margin_proxy', 'comp_spread', 'above_comp_avg', 'rating', 'brand_encoded', 'category_encoded']


In [25]:
# Encode category + save

import os
import json

# Category encoding
cat_list = df['category'].unique().tolist()
cat_map  = {c: i for i, c in enumerate(sorted(cat_list))}
df['category_encoded'] = df['category'].map(cat_map)

# Ensure directories exist
os.makedirs('../data', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# Save processed data
df.to_csv('../data/flipkart_processed.csv', index=False)

# Save maps
with open('../models/category_map.json', 'w') as f:
    json.dump(cat_map, f, indent=2)

print(f"Saved {len(df):,} rows to data/flipkart_processed.csv")
print(f"Category map saved: {len(cat_map)} categories")
print(df[FEATURES + ['discounted_price']].describe())

Saved 18,269 rows to data/flipkart_processed.csv
Category map saved: 15 categories
       comp_avg_price  comp_min_price  comp_max_price  price_vs_comp  \
count    18269.000000    18269.000000    18269.000000   18269.000000   
mean      1514.624792     1503.907001     1527.221796       1.001284   
std       4156.509499     4137.212486     4186.504076       0.049641   
min        120.000000      120.000000      120.000000       0.803571   
25%        350.000000      350.000000      350.000000       1.000000   
50%        549.000000      549.000000      549.000000       1.000000   
75%        999.000000      999.000000      999.000000       1.000000   
max      31795.333333    31768.000000    31838.000000       6.294854   

          price_gap  discount_pct    log_retail  price_per_rating  \
count  18269.000000  18269.000000  18269.000000      18269.000000   
mean      20.638513      0.409409      7.110182        376.300169   
std      391.457873      0.234200      0.931785       1023.39